# OCI Lifecycle Latency Analysis

This notebook reads a **combined benchmark output file** and filters only `lifecycle` experiment trials before analysis.

The filtering boundary is:

```python
trial['experimentName'] == 'lifecycle'
```

It then extracts create, start, and delete latency only from the selected lifecycle trials.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')


In [ ]:
# Both notebooks can read the same combined harness output file.
RESULT_FILE = Path('run.json')
EXPERIMENT_NAME = 'lifecycle'

with RESULT_FILE.open('r', encoding='utf-8') as file:
    harness_result = json.load(file)

all_trials = harness_result.get('trials', [])
selected_trials = [
    result for result in all_trials
    if result.get('trial', {}).get('experimentName') == EXPERIMENT_NAME
]

lifecycle_harness = {
    **harness_result,
    'trials': selected_trials,
}

print('Run ID:', harness_result.get('runId'))
print('All trials:', len(all_trials))
print(f'{EXPERIMENT_NAME.capitalize()} trials selected:', len(selected_trials))
print('Other experiment trials ignored:', len(all_trials) - len(selected_trials))

assert selected_trials, f'No {EXPERIMENT_NAME!r} trials found in {RESULT_FILE}'
assert all(
    result.get('trial', {}).get('experimentName') == EXPERIMENT_NAME
    for result in selected_trials
)


In [ ]:
pd.DataFrame([
    {
        'trial_id': result.get('trial', {}).get('id'),
        'experiment': result.get('trial', {}).get('experimentName'),
        'runtime': result.get('trial', {}).get('runtimeName'),
        'workload': result.get('trial', {}).get('workloadName'),
        'selected': result.get('trial', {}).get('experimentName') == EXPERIMENT_NAME,
    }
    for result in all_trials
])


## Extract lifecycle metrics

The parser prefers `stage.data.latency_ms`. If that field is missing, it falls back to converting the nanosecond `duration` field to milliseconds.

In [ ]:
TARGET_STAGES = ('create_task', 'start_task', 'delete_task')


def get_stage(trial_result: dict, stage_name: str) -> dict | None:
    return next(
        (stage for stage in trial_result.get('runtimeStages', [])
         if stage.get('stage') == stage_name),
        None,
    )


def stage_latency_ms(stage: dict | None) -> float:
    if not stage:
        return np.nan

    data = stage.get('data') or {}
    if data.get('latency_ms') is not None:
        return float(data['latency_ms'])

    if data.get('latency') is not None:
        return float(data['latency']) / 1_000_000

    if stage.get('duration') is not None:
        return float(stage['duration']) / 1_000_000

    return np.nan


def extract_lifecycle_rows(harness: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    invalid_rows = []

    for result in harness.get('trials', []):
        trial = result.get('trial', {})
        base = {
            'trial_id': trial.get('id'),
            'experiment': trial.get('experimentName'),
            'workload': trial.get('workloadName'),
            'runtime': trial.get('runtimeName'),
            'runtime_handler': trial.get('runtimeHandler'),
            'image': trial.get('image'),
            'harness_status': result.get('status'),
        }

        stages = {name: get_stage(result, name) for name in TARGET_STAGES}
        missing = [name for name, stage in stages.items() if stage is None]

        if missing:
            invalid_rows.append({
                **base,
                'reason': f"Missing lifecycle stages: {', '.join(missing)}",
            })
            continue

        create_ms = stage_latency_ms(stages['create_task'])
        start_ms = stage_latency_ms(stages['start_task'])
        delete_ms = stage_latency_ms(stages['delete_task'])

        if any(np.isnan(value) for value in (create_ms, start_ms, delete_ms)):
            invalid_rows.append({
                **base,
                'reason': 'One or more target stages have no usable latency value',
            })
            continue

        stop_stage = get_stage(result, 'stop')
        stop_data = (stop_stage or {}).get('data') or {}

        rows.append({
            **base,
            'create_latency_ms': create_ms,
            'start_latency_ms': start_ms,
            'delete_latency_ms': delete_ms,
            'total_lifecycle_latency_ms': create_ms + start_ms + delete_ms,
            'trial_duration_ms': result.get('duration', np.nan) / 1_000_000,
            'stop_exit_code': stop_data.get('exit_code'),
        })

    return pd.DataFrame(rows), pd.DataFrame(invalid_rows)


lifecycle_df, invalid_df = extract_lifecycle_rows(lifecycle_harness)


## Trial-level lifecycle table

In [ ]:
display_columns = [
    'runtime', 'workload', 'image',
    'create_latency_ms', 'start_latency_ms',
    'delete_latency_ms', 'total_lifecycle_latency_ms',
    'stop_exit_code',
]

lifecycle_df[display_columns].sort_values('runtime')


## Invalid or incomplete trials

In [ ]:
if invalid_df.empty:
    print('No invalid lifecycle trials detected.')
else:
    display(invalid_df)


## Create, start, and delete latency by runtime

Lower is better.

In [ ]:
plot_df = (
    lifecycle_df.set_index('runtime')
    [['create_latency_ms', 'start_latency_ms', 'delete_latency_ms']]
    .rename(columns={
        'create_latency_ms': 'Create',
        'start_latency_ms': 'Start',
        'delete_latency_ms': 'Delete',
    })
)

ax = plot_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('OCI lifecycle command latency by runtime')
ax.set_xlabel('Runtime')
ax.set_ylabel('Latency (ms)')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Per-command charts

Separate charts are useful because create latency can be much larger than start or delete latency and may otherwise compress the smaller values.

In [ ]:
for column, title in [
    ('create_latency_ms', 'Create task latency'),
    ('start_latency_ms', 'Start task latency'),
    ('delete_latency_ms', 'Delete task latency'),
]:
    ordered = lifecycle_df.sort_values(column)
    ax = ordered.plot(
        x='runtime',
        y=column,
        kind='bar',
        legend=False,
        figsize=(8, 4),
    )
    ax.set_title(title)
    ax.set_xlabel('Runtime')
    ax.set_ylabel('Latency (ms)')
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## Total measured lifecycle latency

This is the sum of create, start, and delete latency only. It is not full end-to-end application startup time.

In [ ]:
ordered = lifecycle_df.sort_values('total_lifecycle_latency_ms')
ax = ordered.plot(
    x='runtime',
    y='total_lifecycle_latency_ms',
    kind='bar',
    legend=False,
    figsize=(8, 4),
)
ax.set_title('Total measured OCI lifecycle latency')
ax.set_xlabel('Runtime')
ax.set_ylabel('Create + start + delete (ms)')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## Relative latency against a baseline

For latency, values below 100% are faster than the baseline; values above 100% are slower.

In [ ]:
BASELINE_RUNTIME = 'runc'

if BASELINE_RUNTIME not in set(lifecycle_df['runtime']):
    raise ValueError(f'Baseline runtime {BASELINE_RUNTIME!r} is not present')

baseline = lifecycle_df.loc[lifecycle_df['runtime'] == BASELINE_RUNTIME].iloc[0]
relative_df = lifecycle_df[[
    'runtime', 'create_latency_ms', 'start_latency_ms',
    'delete_latency_ms', 'total_lifecycle_latency_ms'
]].copy()

for metric in [
    'create_latency_ms', 'start_latency_ms',
    'delete_latency_ms', 'total_lifecycle_latency_ms'
]:
    relative_df[f'{metric}_vs_baseline_percent'] = (
        relative_df[metric] / baseline[metric] * 100
    )

relative_df.sort_values('total_lifecycle_latency_ms_vs_baseline_percent')


## Repeated-trial aggregation

For final benchmark conclusions, run each runtime many times. The table below calculates sample count, mean, standard deviation, median, p95, minimum, and maximum.

In [ ]:
def percentile_95(values: pd.Series) -> float:
    return values.quantile(0.95)


aggregate_df = (
    lifecycle_df.groupby('runtime', as_index=False)
    .agg(
        samples=('trial_id', 'count'),
        create_mean_ms=('create_latency_ms', 'mean'),
        create_std_ms=('create_latency_ms', 'std'),
        create_median_ms=('create_latency_ms', 'median'),
        create_p95_ms=('create_latency_ms', percentile_95),
        create_min_ms=('create_latency_ms', 'min'),
        create_max_ms=('create_latency_ms', 'max'),
        start_mean_ms=('start_latency_ms', 'mean'),
        start_std_ms=('start_latency_ms', 'std'),
        start_median_ms=('start_latency_ms', 'median'),
        start_p95_ms=('start_latency_ms', percentile_95),
        start_min_ms=('start_latency_ms', 'min'),
        start_max_ms=('start_latency_ms', 'max'),
        delete_mean_ms=('delete_latency_ms', 'mean'),
        delete_std_ms=('delete_latency_ms', 'std'),
        delete_median_ms=('delete_latency_ms', 'median'),
        delete_p95_ms=('delete_latency_ms', percentile_95),
        delete_min_ms=('delete_latency_ms', 'min'),
        delete_max_ms=('delete_latency_ms', 'max'),
    )
)

aggregate_df


## Mean latency with standard-deviation error bars

This becomes meaningful once you have several trials per runtime.

In [ ]:
metrics = [
    ('create_latency_ms', 'Create task'),
    ('start_latency_ms', 'Start task'),
    ('delete_latency_ms', 'Delete task'),
]

for metric, title in metrics:
    grouped = lifecycle_df.groupby('runtime')[metric].agg(['mean', 'std']).sort_values('mean')
    errors = grouped['std'].fillna(0)

    ax = grouped['mean'].plot(
        kind='bar',
        yerr=errors,
        capsize=4,
        figsize=(8, 4),
    )
    ax.set_title(f'{title} mean latency with standard deviation')
    ax.set_xlabel('Runtime')
    ax.set_ylabel('Latency (ms)')
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## Distribution plots for repeated trials

Box plots help reveal variance, skew, and outliers. They are not very informative with only one observation per runtime.

In [ ]:
for metric, title in metrics:
    grouped_values = [
        group[metric].dropna().values
        for _, group in lifecycle_df.groupby('runtime')
    ]
    labels = list(lifecycle_df.groupby('runtime').groups.keys())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.boxplot(grouped_values, tick_labels=labels)
    ax.set_title(f'{title} latency distribution')
    ax.set_xlabel('Runtime')
    ax.set_ylabel('Latency (ms)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()


## Fair-comparison checks

A runtime comparison is only directly fair when the workload and image are equivalent. This cell highlights differing images or workloads.

In [ ]:
comparison_checks = pd.DataFrame({
    'field': ['workload', 'image'],
    'unique_values': [
        lifecycle_df['workload'].dropna().unique().tolist(),
        lifecycle_df['image'].dropna().unique().tolist(),
    ],
})
comparison_checks['comparable'] = comparison_checks['unique_values'].apply(lambda values: len(values) == 1)
comparison_checks
